<a href="https://colab.research.google.com/github/KatiaItzelCortes/Simulacion-1/blob/main/sistemaEsperaLineaDeDossrvidores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de Colas con Dos Servidores en Serie

Un sistema de servidores en serie (o en tándem) modela una situación donde cada cliente que ingresa al sistema debe recibir servicio en dos estaciones consecutivas de manera secuencial. El flujo es estricto: el cliente llega al servidor 1 y, una vez completado su servicio, pasa inmediatamente a formarse al servidor 2. El cliente sale del sistema solo cuando termina su atención en la segunda estación.

**Elementos Clave del Modelo:**
* **Tiempos de llegada ($t_A$):** Modelados mediante un proceso de Poisson.
* **Tiempos de servicio:** Representados por variables aleatorias $Y_1$ (para el servidor 1) e $Y_2$ (para el servidor 2).
* **Variables de estado (ES):** El par $(n_1, n_2)$, donde $n_1$ es el número de clientes en el servidor 1 (incluyendo a los que esperan y al que está siendo atendido) y $n_2$ es el número de clientes en el servidor 2.
* **Lista de eventos (LE):** Compuesta por $t_A$ (tiempo de la siguiente llegada), $t_1$ (tiempo de salida del servidor 1) y $t_2$ (tiempo de salida del servidor 2). El avance del reloj de simulación depende del $\min(t_A, t_1, t_2)$.

**Casos de Actualización del Sistema:**
La simulación avanza iterando sobre los siguientes tres eventos mutuamente excluyentes:

1. **Caso 1: $\min(t_A, t_1, t_2) = t_A$ (Llegada al sistema).**
   Un cliente ingresa al sistema. Se actualiza el reloj $t = t_A$. Se incrementa el contador de llegadas y la variable de estado $n_1$. Si el servidor 1 estaba desocupado ($n_1 = 1$), se genera el tiempo de su servicio $t_1 = t + Y_1$. Se programa la siguiente llegada $t_A$.

2. **Caso 2: $\min(t_A, t_1, t_2) = t_1$ (Tránsito del Servidor 1 al Servidor 2).**
   Un cliente termina en el servidor 1 y pasa al 2. Se actualiza el reloj $t = t_1$. La variable $n_1$ disminuye y $n_2$ aumenta. Si la fila del servidor 1 queda vacía ($n_1 = 0$), el servidor queda ocioso ($t_1 = \infty$). Si hay más clientes, se programa el siguiente servicio $t_1 = t + Y_1$. Si el servidor 2 estaba ocioso al recibir al cliente ($n_2 = 1$), se programa su tiempo de salida $t_2 = t + Y_2$.

3. **Caso 3: $\min(t_A, t_1, t_2) = t_2$ (Salida del sistema).**
   Un cliente concluye su servicio en el servidor 2 y abandona el sistema. Se actualiza el reloj $t = t_2$. Se incrementa el contador de salidas y la variable de estado $n_2$ disminuye. Si el servidor 2 queda vacío ($n_2 = 0$), se marca como ocioso ($t_2 = \infty$). Si hay fila, entra el siguiente cliente a atención y se programa su salida $t_2 = t + Y_2$.

In [55]:
import random
import pandas as pd

In [56]:
def generar_Tr(tasa_llegadas, tiempo_actual, tiempo_max_sistema):
  if tiempo_actual <= tiempo_max_sistema:
    return tiempo_actual + random.expovariate(tasa_llegadas)
  else:
    return float('inf')

def simulacion_servidores_serie(T_max, tasa_llegadas, tasa_serv_1, tasa_serv_2):
    # Inicialización
    t = 0
    N_A = 0
    N_D = 0
    n1 = 0  # Clientes en servidor 1 (incluyendo fila)
    n2 = 0  # Clientes en servidor 2 (incluyendo fila)

    # Lista de eventos
    t_A = random.expovariate(tasa_llegadas)
    t_1 = float('inf')
    t_2 = float('inf')

    # Variables de salida (diccionarios para rastrear por número de cliente)
    A1 = {}
    A2 = {}
    D = {}

    while t <= T_max or n1 > 0 or n2 > 0:
        # Detener llegadas después de T_max para que la simulación termine
        if t > T_max and t_A != float('inf'):
            t_A = float('inf')

        # Determinar el siguiente evento
        min_t = min(t_A, t_1, t_2)

        if min_t == float('inf'):
            break # Fin de la simulación

        if min_t == t_A:
            # CASO 1
            t = t_A
            N_A += 1
            n1 += 1
            Tr = generar_Tr(tasa_llegadas, t, T_max)
            t_A = Tr

            if n1 == 1:
                Y1 = random.expovariate(tasa_serv_1)
                t_1 = t + Y1

            A1[N_A] = t

        elif t_1 < t_A and t_1 <= t_2:
            # CASO 2
            t = t_1
            n1 -= 1
            n2 += 1

            if n1 == 0:
                t_1 = float('inf')
            else:
                Y1 = random.expovariate(tasa_serv_1)
                t_1 = t + Y1

            if n2 == 1:
                Y2 = random.expovariate(tasa_serv_2)
                t_2 = t + Y2

            # Identificamos qué cliente pasa al servidor 2
            A2[N_A - n1] = t

        elif t_2 < t_A and t_2 < t_1:
            # CASO 3
            t = t_2
            N_D += 1
            n2 -= 1

            if n2 == 0:
                t_2 = float('inf')
            else:
                Y2 = random.expovariate(tasa_serv_2)
                t_2 = t + Y2

            D[N_D] = t

    return A1, A2, D



In [57]:
# Parámetros: Tiempo=100, Lambda=2, Mu1=3, Mu2=4
Llegada_S1, Llegada_S2, Salida_Sitema = simulacion_servidores_serie(100, 2.0, 3.0, 4.0)

# Consolidar resultados en un DataFrame para visualización
df_serie = pd.DataFrame({
    'Llegada_S1': pd.Series(Llegada_S1),
    'Llegada_S2': pd.Series(Llegada_S2),
    'Salida_Sistema': pd.Series(Salida_Sitema)
})

# Calcular tiempo total en el sistema
df_serie['Tiempo_Total_Sistema'] = df_serie['Salida_Sistema'] - df_serie['Llegada_S1']
df_serie.head()

,Llegada_S1,Llegada_S2,Salida_Sistema,Tiempo_Total_Sistema
1,0.550582,1.050077,1.244753,0.694171
2,1.621995,1.685191,1.698959,0.076964
3,2.428910,2.551395,2.675766,0.246856
4,3.372102,3.442372,3.640270,0.268168
5,3.435947,3.937168,3.973522,0.537575


# Sistema de Colas con Dos Servidores en Paralelo


Un sistema de servidores en paralelo modela un escenario donde los clientes llegan a un sistema para recibir un mismo tipo de servicio. Existe una única fila de espera compartida, pero dos servidores (estaciones) independientes atendiendo simultáneamente. Al llegar, un cliente es atendido por el servidor 1 si está libre, o por el servidor 2 si el primero está ocupado. Si ambos están ocupados, el cliente se forma en la fila.

Como los tiempos de servicio varían entre servidores, el orden en el que los clientes abandonan el sistema puede no coincidir con su orden de llegada.

**Elementos Clave del Modelo:**
* **Variables de estado (ES):** La tupla $(n, i_1, i_2, \dots, i_n)$.
    * $n$: Número total de clientes en el sistema.
    * $i_1, i_2$: Identificadores de los clientes siendo atendidos por los servidores 1 y 2, respectivamente.
    * $i_3, i_4, \dots, i_n$: Identificadores de los clientes formados en la fila.
* **Lista de eventos (LE):** Compuesta por $t_A$ (tiempo de siguiente llegada), $t_1$ (tiempo de salida del servidor 1) y $t_2$ (tiempo de salida del servidor 2). El motor de simulación busca el $\min(t_A, t_1, t_2)$.

**Casos de Actualización del Sistema:**
La evolución del sistema se rige por los siguientes escenarios:

1. **Caso 1: $t_A = \min(t_A, t_1, t_2)$ (Llegada al sistema).**
   Un nuevo cliente ingresa. Se actualiza el reloj $t = t_A$. Se registra su llegada y se programa la siguiente ($t_A$). El estado se actualiza dependiendo de la ocupación:
   * **Si el sistema estaba vacío ($ES = (0)$):** El cliente pasa al servidor 1, $t_1 = t + Y_1$.
   * **Si el servidor 1 estaba ocupado pero el 2 libre ($ES = (1, j, 0)$):** El cliente pasa al servidor 2, $t_2 = t + Y_2$.
   * **Si el servidor 2 estaba ocupado pero el 1 libre ($ES = (1, 0, j)$):** El cliente pasa al servidor 1, $t_1 = t + Y_1$.
   * **Si ambos están ocupados ($n \ge 2$):** El cliente se añade al final de la fila de espera ($i_n$).

2. **Caso 2: $t_1 = \min(t_A, t_1, t_2)$ (Salida del servidor 1).**
   Un cliente termina de ser atendido en el servidor 1 y sale del sistema. Se actualiza el reloj $t = t_1$.
   * **Si no hay fila ($n \le 2$):** El servidor 1 queda desocupado ($t_1 = \infty$).
   * **Si hay clientes en fila ($n > 2$):** El cliente $i_3$ (el primero en la fila) pasa a ser atendido por el servidor 1. Se genera su tiempo de salida $t_1 = t + Y_1$. El resto de la fila avanza una posición.

3. **Caso 3: $t_2 = \min(t_A, t_1, t_2)$ (Salida del servidor 2).**
   Un cliente termina su servicio en el servidor 2 y abandona el sistema. Se actualiza el reloj $t = t_2$. El comportamiento es análogo al Caso 2:
   * **Si no hay fila:** El servidor 2 queda desocupado ($t_2 = \infty$).
   * **Si hay fila:** El primer cliente de la fila ingresa a atención en el servidor 2, generándose su tiempo de salida $t_2 = t + Y_2$.

In [58]:
import random as r

In [59]:
def generar_T0():
  # genera una numero aleatorio con una distribución uniforme
  return r.random()


In [61]:
def servidores_paralelo(T_max, tasa_llegadas, tasa_serv_1, tasa_serv_2):
    # Inicialización
    t = 0
    N_A = 0
    C1 = 0 # Clientes atendidos por serv 1
    C2 = 0 # Clientes atendidos por serv 2

    # Estado del sistema ES
    n = 0
    i1 = 0
    i2 = 0
    fila = [] # Contendrá los clientes en espera i_3, i_4...

    # Lista de eventos
    t_A = generar_T0()
    t_1 = float('inf')
    t_2 = float('inf')

    A = {}
    D = {}

    while t <= T_max or n > 0:
        if t > T_max and t_A != float('inf'):
            t_A = float('inf')

        min_t = min(t_A, t_1, t_2)

        if min_t == float('inf'):
            break

        if min_t == t_A:
            # CASO 1:
            t = t_A
            N_A += 1
            A[N_A] = t
            t_A = t + random.expovariate(tasa_llegadas) if t <= T_max else float('inf')

            if n == 0:
                # Sistema vacío, pasa al servidor 1 por defecto
                n = 1
                i1 = N_A
                Y1 = random.expovariate(tasa_serv_1)
                t_1 = t + Y1
            elif n == 1 and i1 != 0 and i2 == 0:
                # S1 ocupado, pasa al servidor 2
                n = 2
                i2 = N_A
                Y2 = random.expovariate(tasa_serv_1)
                t_2 = t + Y2
            elif n == 1 and i1 == 0 and i2 != 0:
                # S2 ocupado, pasa al servidor 1
                n = 2
                i1 = N_A
                Y1 = random.expovariate(tasa_serv_1)
                t_1 = t + Y1
            elif n > 1:
                # Ambos ocupados, se forma en la fila
                n += 1
                fila.append(N_A)

        elif n != 0 and t_1 < t_A and t_1 <= t_2:
            # CASO 2
            t = t_1
            C1 += 1
            D[i1] = t

            if n == 1:
                # Si no quedan clientes, o el que queda está en S2
                n = 0
                i1 = 0
                i2 = 0
                fila.clear()
                t_1 = float('inf')
            elif n == 2:
                n = 1
                i1 = 0
                t_1 = float('inf')
            else: # n >= 2
                n -= 1
                i1 = fila.pop(0)
                Y1 = random.expovariate(tasa_serv_1)
                t_1 = t + Y1

        elif n != 0 and t_2 < t_A and t_2 < t_1:
            # CASO 3
            t = t_2
            C2 += 1
            D[i2] = t

            if n == 1:
                # Si no quedan clientes, o el que queda está en S1
                n = 0
                i1 = 0
                i2 = 0
                fila.clear()
                t_2 = float('inf')
            elif n == 2:
                n = 1
                i2 = 0
                t_2 = float('inf')
            else: # n > 2 (hay alguien en la fila)
                n -= 1
                i2 = fila.pop(0)
                Y2 = random.expovariate(tasa_serv_2)
                t_2 = t + Y2

    return A, D

In [62]:
# Parámetros: Tiempo=100, Lambda=5, Mu1=3, Mu2=4
Llegadas, Salidas = servidores_paralelo(100, 5.0, 3.0, 4.0)
df_paralelo = pd.DataFrame({
    'Llegada': pd.Series(Llegadas),
    'Salida': pd.Series(Salidas)
})

df_paralelo['Tiempo_En_Sistema'] = df_paralelo['Salida'] - df_paralelo['Llegada']
df_paralelo.head()

,Llegada,Salida,Tiempo_En_Sistema
1,0.493966,0.624464,0.130499
2,1.233273,1.483457,0.250184
3,1.427986,1.438989,0.011003
4,2.040067,2.567471,0.527404
5,2.084122,2.133797,0.049675
